# Session 2 — Balanced Multilingual BPE Tokenizer

Train a single **10,000-token** BPE tokenizer on the Wikipedia *India*
article in **English, Hindi, Telugu and Kannada** such that each
language's fertility ratio

$$X_i = \frac{\text{tokens produced on language } i}{\text{whitespace-separated words}}$$

is **≤ 1.2** and the four ratios are as close as possible.

**Score** $= 1000 / (X_{max} - X_{min})$

Design: codepoint-level BPE, whitespace attached to the following word,
and a *balanced merge loop* — every merge is taken from whichever
language currently has the worst fertility, which greedily minimizes
$X_{max} - X_{min}$ at every step.

**Corpus size:** each language uses the first **2,000 words** of its
article (Kannada's whole article is 1,019 words). With the full
articles the fertility floor at 10k vocab is ≈ 1.46 — measured, not
guessed — so X ≤ 1.2 is infeasible; at 2,000 words per language all
four X land around 1.03.

In [1]:
import subprocess, sys
from pathlib import Path
from bpe_tokenizer import BalancedBPETokenizer
from train_and_evaluate import WORD_CAP, load_corpora

LANGS = {'en': 'English', 'hi': 'Hindi', 'te': 'Telugu', 'kn': 'Kannada'}
DATA = Path('data')

# Download the four corpora if not already present
if not all((DATA / f'{l}_india.txt').exists() for l in LANGS):
    subprocess.run([sys.executable, 'download_data.py'], check=True)

corpora = load_corpora()  # capped at WORD_CAP words per language
print('word cap per language:', WORD_CAP)
print(f"{'lang':<10}{'full words':>12}{'used words':>12}"
      f"{'used chars':>12}{'unique chars':>14}")
for l, text in corpora.items():
    full = (DATA / f'{l}_india.txt').read_text(encoding='utf-8')
    print(f'{LANGS[l]:<10}{len(full.split()):>12,}{len(text.split()):>12,}'
          f'{len(text):>12,}{len(set(text)):>14,}')

word cap per language: 2000
lang        full words  used words  used chars  unique chars
English         10,121       2,000      13,067            98
Hindi            8,078       2,000      10,782           107
Telugu           2,511       2,000      16,267            83
Kannada          1,019       1,019       7,740           108


## Train (or load) the tokenizer

Training takes only a few seconds on these capped corpora, so the
notebook trains the tokenizer live by default (`RETRAIN = True`),
reproducing the committed artifact `tokenizer_10k.json`
deterministically. Set `RETRAIN = False` to load the committed
artifact instead (equivalent to `python3 train_and_evaluate.py`).

In [2]:
RETRAIN = True
if RETRAIN or not Path('tokenizer_10k.json').exists():
    tok = BalancedBPETokenizer.train(corpora, vocab_size=10_000,
                                     verbose=True)
    tok.save('tokenizer_10k.json')
else:
    tok = BalancedBPETokenizer.load('tokenizer_10k.json')
print('vocab size:', tok.vocab_size)
print('base codepoints:', len(tok.base_chars))
print('learned merges:', len(tok.merges))

merge 1000/9705: fertility={'en': 3.268, 'hi': 3.2705, 'te': 3.273, 'kn': 3.2728}


merge 2000/9705: fertility={'en': 2.5625, 'hi': 2.562, 'te': 2.564, 'kn': 2.5623}


merge 3000/9705: fertility={'en': 2.156, 'hi': 2.156, 'te': 2.1565, 'kn': 2.156}


merge 4000/9705: fertility={'en': 1.926, 'hi': 1.9255, 'te': 1.926, 'kn': 1.9254}


merge 5000/9705: fertility={'en': 1.727, 'hi': 1.727, 'te': 1.727, 'kn': 1.7272}


merge 6000/9705: fertility={'en': 1.5605, 'hi': 1.56, 'te': 1.5605, 'kn': 1.5604}
merge 7000/9705: fertility={'en': 1.416, 'hi': 1.416, 'te': 1.416, 'kn': 1.4151}


merge 8000/9705: fertility={'en': 1.2725, 'hi': 1.2725, 'te': 1.2725, 'kn': 1.2718}
merge 9000/9705: fertility={'en': 1.1295, 'hi': 1.13, 'te': 1.13, 'kn': 1.1295}
vocab size: 10000
base codepoints: 294
learned merges: 9705


## Per-language fertility and score

In [3]:
results = {}
for l, text in corpora.items():
    ids = tok.encode(text)
    assert tok.decode(ids) == text, f'round-trip failed for {l}'
    words = len(text.split())
    results[l] = (words, len(ids), len(ids) / words)

print(f"{'lang':<10}{'words':>10}{'tokens':>10}{'X (tok/word)':>15}")
for l, (w, t, x) in results.items():
    flag = 'OK' if x <= 1.2 else 'FAIL'
    print(f'{LANGS[l]:<10}{w:>10,}{t:>10,}{x:>15.4f}  {flag}')

xs = [x for _, _, x in results.values()]
spread = max(xs) - min(xs)
print(f'\nX_max - X_min = {spread:.6f}')
print(f'score = 1000 / (X_max - X_min) = '
      f"{1000 / spread:,.1f}" if spread > 0 else 'score = inf')

lang           words    tokens   X (tok/word)
English        2,000     2,058         1.0290  OK
Hindi          2,000     2,058         1.0290  OK
Telugu         2,000     2,059         1.0295  OK
Kannada        1,019     1,049         1.0294  OK

X_max - X_min = 0.000500
score = 1000 / (X_max - X_min) = 2,000,000.0


## Encode / decode demo

In [4]:
samples = {
    'en': 'India is the seventh-largest country in the world.',
    'hi': 'भारत दक्षिण एशिया में स्थित एक देश है।',
    'te': 'భారతదేశం ప్రపంచంలో ఏడవ పెద్ద దేశం.',
    'kn': 'ಭಾರತವು ದಕ್ಷಿಣ ಏಷ್ಯಾದಲ್ಲಿರುವ ಒಂದು ದೇಶ.',
}
for l, s in samples.items():
    ids = tok.encode(s)
    pieces = [tok.id_to_str[i] for i in ids]
    assert tok.decode(ids) == s
    print(f'{LANGS[l]}: {len(s.split())} words -> {len(ids)} tokens')
    print('  ', pieces, '\n')

English: 8 words -> 10 tokens
   ['In', 'dia', ' is', ' the', ' seventh-largest', ' country', ' in', ' the', ' world', '.'] 

Hindi: 8 words -> 8 tokens
   ['भारत', ' दक्षिण', ' एशिया', ' में', ' स्थित', ' एक', ' देश', ' है।'] 

Telugu: 5 words -> 7 tokens
   ['భారతదేశం', ' ప్రపంచంలో', ' ఏ', 'డ', 'వ', ' పెద్ద', ' దేశం.'] 

Kannada: 5 words -> 9 tokens
   ['ಭಾರತ', 'ವು', ' ದಕ್ಷಿಣ', ' ಏ', 'ಷ್ಯಾ', 'ದಲ್ಲಿ', 'ರುವ', ' ಒಂದು', ' ದೇಶ.'] 

